## Homework: Evaluation

In homework 2 we built keyword, vector, and hybrid search over the course
lessons, and ended with an open question: which one is best? The way to answer
that is to measure, and that's what we do here.

In this homework we generate a ground truth dataset and use it to evaluate
search, the same way we did in the module. There we only evaluated keyword
search. Here we also evaluate vector and hybrid search, so we can finally
compare them on numbers instead of intuition.

Like in homework 1 and 2, our knowledge base is the course lessons themselves.
Each module has a `lessons/` folder of numbered markdown pages, and we pull
them from GitHub. We use commit `8c1834d`, so everyone works with the exact
same 72 pages.

> It's possible your answers won't match exactly. If so, select the closest one.


## Setup
<details>

This homework continues from homework 2. We reuse the same chunks and the same
search functions, so it's easiest to keep working in the same project.

We need a few more libraries for generating questions with an LLM:

```bash
uv add openai pydantic python-dotenv pandas
```

For the LLM, we recommend OpenAI with `gpt-5.4-mini`, but you can use any model
and provider you like - just adapt the client accordingly. Put your key in a
`.env` file as in the earlier modules.

Load the data exactly as in homework 2:

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
```

This gives 72 pages.
</details>

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
documents.__len__()

72

## Generating ground truth

<details>
To evaluate search, we need a dataset of questions where we know which document
is the correct answer. This is the ground truth.

We generate it the same way as in the module. For each lesson page, we ask an
LLM to write 5 questions that are answered by that page. Each question is then
labeled with the page it came from.

We use the same structured-output approach as in the module - the same
`Questions` model and the `llm_structured` helper from `evaluation_utils.py`.

Download `evaluation_utils.py` and the `rag_helper.py` it depends on:
```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/04-evaluation/code/evaluation_utils.py
```

</details>

The module's instructions generate questions from a FAQ record, so we adapt
them for a lesson page:


In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

We ask for different wording from the page on purpose. Real users don't phrase
their questions the way the lesson does, and copying the text would make the
evaluation too easy.

For each page, build a JSON user prompt from its `filename` and `content`, then
call `llm_structured` with the `Questions` model. Turn each returned question
into a record labeled with the page's `filename`. The call also returns the
token usage, the same as in the lessons.

The output as a list of strings, so we define that structure
with a Pydantic model:

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

 ## Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's
start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

Each call returns the token usage, which most LLM APIs report on the response
object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?


In [5]:
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

Call the model:

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
import json
from evaluation_utils import llm_structured

responses = []

for doc in documents[:3]: # Get the messages from the docs
    user_prompt = json.dumps(doc)
    #user_prompt

    messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ]

    responses.append(llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    ))



In [8]:
from evaluation_utils import llm_structured
user_prompt = json.dumps(documents[0])
messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ]
response, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

In [9]:
response

ParsedResponse[~TextFormatT](id='resp_00fbd130af44f578006a5534a82d80819394c350ca4f646f7a', created_at=1783968936.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ParsedResponseOutputMessage[~TextFormatT](id='msg_00fbd130af44f578006a5534ab11508193a74f36761e216ddf', content=[ParsedResponseOutputText[~TextFormatT](annotations=[], text='{"questions":["What is the main idea behind retrieval-augmented generation, and why does it help compared with just asking the model directly?","Why does the course treat the language model like a black box instead of studying how it works inside?","What kinds of problems do large language models have that RAG is supposed to fix?","What project will be built in this module to show RAG in practice?","What will be covered in the first part of the module versus the second part?"]}', type='output_text', logprobs=[], parsed=Questions(questions=['What is the main idea behind retrie

In [10]:
usage

ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1129)

In [11]:
sum = 0
for _, usage in responses:
    sum = usage.input_tokens + sum
print(f"average : {sum/3}")

average : 1353.0


## The full ground truth

You don't need to generate the data for the rest of the homework. We already
did it for all 72 pages, using the same approach as in the lessons, and saved
the 360 questions to a file.

Download it:

In [12]:
PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main'
!wget {PREFIX}'/cohorts/2026/04-evaluation/ground-truth.csv'

--2026-07-13 12:55:40--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8003::154, 2606:50c0:8000::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8003::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv.1’

ground-truth.csv.1  100%[===================>]  47.49K  --.-KB/s    in 0.05s   

2026-07-13 12:55:40 (962 KB/s) - ‘ground-truth.csv.1’ saved [48627/48627]



Load it with pandas into a list of records called `ground_truth`. Each record
has a `question` and the `filename` of the page that should answer it.

## Searching the chunks

We search over the same chunks as in homework 2.

Create them with `chunk_documents`:

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
chunks.__len__()

295

In [15]:
chunks[:2]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

This gives 295 chunks.

Now rebuild the search from homework 2 over these chunks. Build a text index
(`Index`) and a vector index (`VectorSearch`), both keyed on `filename`. Wrap
each one in a function, `text_search` and `vector_search`, that takes a query
and the number of results to return (5 by default).

For hybrid search, reuse the `rrf` function from homework 2:

In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Creating *Text Search*:

In [49]:
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [50]:
index = build_index(chunks)

In [51]:
def text_search(query, num_results):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

Creating *Vector Search*:

In [70]:
texts = [content['content'] for content in chunks]

In [71]:
texts[:3]

['# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language

We also need two helper scripts from the embed/ directory of the course repo:

In [25]:
PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed'
!wget {PREFIX}'/download.py'

--2026-07-13 12:59:28--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8003::154, 2606:50c0:8000::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8003::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

download.py         100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-07-13 12:59:29 (57.1 MB/s) - ‘download.py’ saved [1376/1376]



In [26]:
!uv run python download.py

  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [27]:
from embedder import Embedder

embed = Embedder()

In [72]:
from re import X
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [73]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [66]:
def vector_search(query, num_results):
    v_embed = embed.encode(query)
    return vindex.search(
        v_embed, 
        num_results=num_results, 
    )

Then define `hybrid_search` on top of it:


In [32]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

Take the first question from the ground truth:

```python
q = ground_truth[0]["question"]
```

After running `text_search` for it, what's the `filename` of the first result?

Loading the `ground-truth.csv` using pandas to get the data frame into *list*:

In [39]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [56]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [41]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [55]:
text_search(q, num_results=1)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

<details>
<summary>NOTE:</summary>

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice
that one method finds the right page at the top and the other doesn't. That's
exactly why we measure across the whole dataset instead of trusting one query.
</details>

In [75]:
vector_search(q, num_results=1)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph